# 03 - Preprocessing and Datasets Split

## Data sources
- **ML-HydPARK** (Zenodo, v0.0.5): cleaned experimental thermodynamic data for metal hydrides (~770 entries)
- **ElementalH_Ef** and **ElementalH_Ef_MP**: elemental features from Materials Project, sourced from HEAhydrideML (Witman et al.)

## Target Variables
- `Temperature_oC`
- `Hydrogen_Weight_Percent`

## Goals
- Handling missing values
- Feature selection and engineering
- Split into Dataset A and Dataset B
- Per-dataset: encoding, scaling and train/val/test split

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Loading the dataset
df = pd.read_csv('../data/processed/ML-HYDPARK_eda_cleaned.csv')
df.shape

(772, 9)

In [3]:
df.head()

,Material_Class,Composition_Formula,Hydrogen_Weight_Percent,Heat_of_Formation_kJperMolH2,Temperature_oC,Pressure_Atmospheres_Absolute,Entropy_of_Formation_JperMolH2perK,LnEquilibrium_Pressure_25C,HtoM
0,A2B,Th2Al,0.8,130.0,500.0,0.001,110.711134,-39.127349,1.309571
1,A2B,Ti2Cu,2.2,130.0,500.0,0.120,150.515102,-34.339857,1.184850
2,A2B,Zr2Cu,1.3,144.0,600.0,0.003,116.621978,-44.064155,1.071443
3,A2B,Zr2Ni,1.3,183.0,604.0,0.003,160.332084,-54.539843,1.050307
4,A2B,Mg2Ni,3.6,64.5,299.0,3.200,122.403296,-11.297687,1.325126


## Dropping of features

`LnEquilibrium_Pressure_25C`: it is redundant as it can be calculated from `Heat_of_Formation_kJperMolH2` and `Entropy_of_Formation_JperMolH2perK` via the van 't Hoff equation.

`HtoM`: it is mathematically derivable from `Hydrogen_Weight_Percent`, so keeping both when one is the target variable would introduce data leakage.

In [4]:
# Dropping 'LnEquilibrium_Pressure_25C'
df = df.drop(columns = ['LnEquilibrium_Pressure_25C'], axis = 1)
df.shape

(772, 8)

In [5]:
# Dropping 'HtoM'
df = df.drop(columns = ['HtoM'], axis = 1)
df.shape

(772, 7)

In [6]:
df.head()

,Material_Class,Composition_Formula,Hydrogen_Weight_Percent,Heat_of_Formation_kJperMolH2,Temperature_oC,Pressure_Atmospheres_Absolute,Entropy_of_Formation_JperMolH2perK
0,A2B,Th2Al,0.8,130.0,500.0,0.001,110.711134
1,A2B,Ti2Cu,2.2,130.0,500.0,0.120,150.515102
2,A2B,Zr2Cu,1.3,144.0,600.0,0.003,116.621978
3,A2B,Zr2Ni,1.3,183.0,604.0,0.003,160.332084
4,A2B,Mg2Ni,3.6,64.5,299.0,3.200,122.403296


## Handling Missing Values

In [7]:
# Checking missing values across the cleaned dataset
df.isna().sum()

Material_Class                         25
Composition_Formula                     0
Hydrogen_Weight_Percent                31
Heat_of_Formation_kJperMolH2            0
Temperature_oC                        360
Pressure_Atmospheres_Absolute         369
Entropy_of_Formation_JperMolH2perK      0
dtype: int64

In [8]:
# Further exploring the dataset in order to impute the missing values for 'Material_Class'
df.sample(20)

,Material_Class,Composition_Formula,Hydrogen_Weight_Percent,Heat_of_Formation_kJperMolH2,Temperature_oC,Pressure_Atmospheres_Absolute,Entropy_of_Formation_JperMolH2perK
95,AB2,Zr1Mn1.22Fe1.11,1.500000,13.000,23.0,1.00,43.896674
533,AB2(C14),(Ti0.41Zr0.59)(Mn0.44Fe0.56)1.88,1.575803,26.910,NaN,NaN,104.260000
579,AB2(C14),(Ti0.85Zr0.15)(Cr0.42Mn0.5Mo0.08)1.82,1.662106,21.700,NaN,NaN,100.900000
419,NaN,TiCrMn,2.100000,19.600,NaN,NaN,106.000000
39,AB,Hf0.3Zr0.7Co1,1.400000,80.300,200.0,0.03,140.559542
714,AB2(C15),(Ti0.15Zr0.85)(V0.06Mn0.33Fe0.06Ni0.56)2.11,1.510021,28.129,NaN,NaN,98.060000
574,AB2(C14),(Ti0.83Zr0.17)(V0.05Cr0.48Fe0.48)1.96,1.518279,20.380,NaN,NaN,102.840000
446,AB2(C14),(Ti)(Cr0.48Mn0.15Fe0.38)1.96,1.831567,18.700,NaN,NaN,116.400000
646,AB2(C14),(Zr)(Cr0.36Mn0.29Fe0.36)2.8,1.400167,29.500,NaN,NaN,94.800000
418,NaN,TiCr1.5Mn0.2Fe0.3,NaN,18.320,NaN,NaN,101.000000


### Imputing `Material_Class`

25 entries have no `Material_Class` label. Rather than dropping them, we assign classes manually by inspecting the `Composition_Formula` and applying the stoichiometric A:B ratio rule:
- Group elements into A-site (larger atoms) and B-site (smaller transition metals)
- The total A:B atom count determines the class (1:5 → AB5, 1:2 → AB2, 1:1 → AB)
- V-based alloys with no clean integer ratio are solid solutions (SS)
- AB2 subtype (C14 hexagonal vs C15 cubic Laves) is assigned based on B-site chemistry: Mn-dominated → C14, Fe/Cr-dominated with ZrFe2 or TiCr2 parent → C15

Classes were verified against existing entries in the dataset and cross-checked with literature on Laves phase stability.

In [9]:
# Manual imputation of Material_Class for 25 entries with missing labels
# Assignment based on A:B stoichiometric ratio and Laves phase B-site chemistry
class_map = {
    # AB5 - La/Ce/Y on A-site, Ni+substituents sum to 5 on B-site
    'LaNi4.7Sn0.3':          'AB5',
    'LaNi4.8Sn0.2':          'AB5',
    'LaNi4.8Al0.2':          'AB5',
    'La0.85Ce0.15Ni5':       'AB5',
    'La0.2Y0.8Ni4.6Mn0.4':  'AB5',
    'La0.4Ce0.4Ca0.2Ni5':   'AB5',
    # AB - 1:1 ratio
    'TiFe0.9Mn0.1':          'AB',
    # SS - V-based BCC solid solutions, no ordered A/B sublattice
    'V75Ti17.5Zr7.5':        'SS',
    'V75Ti10Zr7.5Cr7.5':     'SS',
    'V92.5Zr7.5':            'SS',
    'V0.85Ti0.1Fe0.05':      'SS',
    # AB2(C15) - TiCr2 or ZrFe2 parent, Cr/Fe-dominated B-site, cubic Laves
    'TiCr1.9':                        'AB2(C15)',
    'TiCr1.9Mo0.01':                  'AB2(C15)',
    'Ti0.86Mo0.14Cr1.9':              'AB2(C15)',
    'ZrFe1.8Cr0.2':                   'AB2(C15)',
    'ZrFe1.8Ni0.2':                   'AB2(C15)',
    'Zr0.8Ti0.2FeNi0.8V0.2':         'AB2(C15)',
    # AB2(C14) - Mn-dominated B-site or significant Mn substitution, hexagonal Laves
    'TiCrMn':                                          'AB2(C14)',
    'Ti0.8Zr0.2CrMn':                                 'AB2(C14)',
    'TiCr1.5Mn0.25Fe0.25':                            'AB2(C14)',
    'TiCr1.5Mn0.2Fe0.3':                              'AB2(C14)',
    '(Ti0.97Zr0.03)1.1Cr1.6Mn0.4':                   'AB2(C14)',
    'Zr0.7Ti0.3Mn2':                                  'AB2(C14)',
    'Ti0.9Zr0.1Mn1.4Cr0.35V0.2Fe0.05':               'AB2(C14)',
    'Ti0.77Zr0.3Cr0.85Fe0.7Mn0.25Ni0.2Cu0.03':       'AB2(C14)',
}

for formula, cls in class_map.items():
    df.loc[df['Composition_Formula'] == formula, 'Material_Class'] = cls

df['Material_Class'].isna().sum()

np.int64(0)

## Dataset Split

In [10]:
# Splitting the dataset into Dataset A and Dataset B
# Target variables are 'Hydrogen_Weight_Percent' and 'Temperature_oC' respectively
df_A = df.dropna(subset=['Hydrogen_Weight_Percent']).copy()
df_B = df.dropna(subset=['Temperature_oC']).copy()

print(df_A.shape)
print(df_B.shape)

(741, 7)
(412, 7)


In [11]:
# Checking NaNs in each dataset
print(df_A.isna().sum())
print("---")
print(df_B.isna().sum())

Material_Class                          0
Composition_Formula                     0
Hydrogen_Weight_Percent                 0
Heat_of_Formation_kJperMolH2            0
Temperature_oC                        351
Pressure_Atmospheres_Absolute         360
Entropy_of_Formation_JperMolH2perK      0
dtype: int64
---
Material_Class                         0
Composition_Formula                    0
Hydrogen_Weight_Percent               22
Heat_of_Formation_kJperMolH2           0
Temperature_oC                         0
Pressure_Atmospheres_Absolute          9
Entropy_of_Formation_JperMolH2perK     0
dtype: int64


### Handling remaining NaNs per dataset

**Dataset A** (`Hydrogen_Weight_Percent` as target):
- `Temperature_oC`: 351/741 = 47% missing. Too sparse to impute reliably, column will be dropped. It is also the target variable of Dataset B, so using it as a feature here would be conceptually problematic.
- `Pressure_Atmospheres_Absolute`: 360/741 = 49% missing. Same reasoning, will be dropped.

**Dataset B** (`Temperature_oC` as target):
- `Hydrogen_Weight_Percent`: 22/412 = 5% missing. Rows to be dropped.
- `Pressure_Atmospheres_Absolute`: 9/412 = 2% missing. Rows to be dropped.

In [ ]:
# Dropping 'Temperature_oC' and 'Pressure_Atmospheres_Absolute' from df_A
df_A = df_A.drop(columns = ['Temperature_oC', 'Pressure_Atmospheres_Absolute'], axis = 1)
df_A.shape

(741, 5)

In [ ]:
# Dropping rows where Hydrogen_Weight_Percent or Pressure_Atmospheres_Absolute is null in df_B
df_B = df_B.dropna(subset=['Hydrogen_Weight_Percent', 'Pressure_Atmospheres_Absolute'])
df_B.shape

(381, 7)

### Cleaned Datasets Overview

Dataset A: 741 rows, 5 columns. Target: `Hydrogen_Weight_Percent`

Dataset B: 381 rows, 7 columns. Target: `Temperature_oC`